# M3L4 E09 — LangGraph router + Langfuse
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

**Objetivo:** crear un router multiagente en LangGraph y ver en Langfuse exactamente qué nodo se ejecutó para cada consulta.

## Arquitectura
```
START
  ↓
router_node
  ├── hr_node
  ├── it_node
  ├── finance_node
  ├── legal_node
  └── general_node
        ↓
       END
```

## Lo que verás en Langfuse
- Una trace por ejecución del grafo
- Un span por cada nodo ejecutado
- `metadata.langfuse_tags` para filtrar ejecuciones
- `metadata.langfuse_user_id` para identificar el usuario

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph
print('Instalación completa.')

In [ ]:
import os
from getpass import getpass

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')
print('Credenciales OK.')

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langfuse.langchain import CallbackHandler

print('Imports OK.')

## Router v2 (reutilizado de E05)

In [ ]:
def route_query_v2(query: str) -> str:
    q = query.lower()
    hr_kw      = ['vacaciones', 'licencia', 'recibo', 'nómina', 'rrhh']
    it_kw      = ['vpn', 'error', 'app', 'laptop', 'wifi', 'login', 'contraseña']
    finance_kw = ['factura', 'pago', 'reembolso', 'gasto', 'cobro', 'comprobante', 'salario']
    legal_kw   = ['contrato', 'legal', 'confidencialidad', 'nda', 'acuerdo']
    detected = []
    if any(w in q for w in hr_kw):      detected.append('hr')
    if any(w in q for w in it_kw):      detected.append('it')
    if any(w in q for w in finance_kw): detected.append('finance')
    if any(w in q for w in legal_kw):   detected.append('legal')
    if len(detected) > 1:   return 'multi_intent'
    if len(detected) == 1:  return detected[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

print('Router v2 listo.')

## Estado

In [ ]:
class AgentState(TypedDict):
    query: str
    intent: str
    response: str

print('AgentState definido.')

## TODO — Nodos del sistema

Implementá los 6 nodos: `router_node` y los 5 nodos de agente.

In [ ]:
def router_node(state: AgentState) -> dict:
    """
    Detecta el intent de la query usando route_query_v2.
    Retorna solo {'intent': intent}
    """
    # TODO
    pass

def hr_node(state: AgentState) -> dict:
    # TODO: retornar respuesta de HR
    pass

def it_node(state: AgentState) -> dict:
    # TODO: retornar respuesta de IT
    pass

def finance_node(state: AgentState) -> dict:
    # TODO: retornar respuesta de Finance
    pass

def legal_node(state: AgentState) -> dict:
    # TODO: retornar respuesta de Legal
    pass

def general_node(state: AgentState) -> dict:
    # TODO: retornar respuesta general
    pass

print('Nodos definidos.')

## TODO — Función condicional

Implementá la función que decide a qué nodo ir según el intent.

In [ ]:
def route_to_node(state: AgentState) -> str:
    """
    Mapea state['intent'] al nombre del nodo destino.
    Cualquier intent no reconocido → 'general_node'
    """
    # TODO
    pass

print('Función condicional definida.')

## TODO — Compilar el grafo

In [ ]:
# TODO: construir el StateGraph con AgentState
# 1. Agregar todos los nodos
# 2. set_entry_point('router_node')
# 3. add_conditional_edges de router_node usando route_to_node
#    con mapping de todos los posibles retornos a sus nodos
# 4. add_edge de cada nodo especialista a END
# 5. Compilar

graph = None  # reemplazar
print('Grafo compilado.')

In [ ]:
if graph:
    print(graph.get_graph().draw_mermaid())

## Ejecutar con Langfuse

In [ ]:
queries = [
    '¿Cómo solicito mis días de vacaciones?',
    'Mi VPN no conecta desde ayer',
    'Necesito ver mi factura del mes pasado',
    'Necesito el contrato de confidencialidad actualizado',
    'ayuda'
]

for i, q in enumerate(queries):
    langfuse_handler = CallbackHandler()

    # TODO: invocar el grafo con:
    # - input: {'query': q, 'intent': '', 'response': ''}
    # - config con callbacks y metadata (tags: ['m3l4', 'router-demo'], user_id: f'student-{i}')
    output = None  # reemplazar

    if output:
        print(f'Query: {q[:45]}')
        print(f'  Intent: {output["intent"]} | Respuesta: {output["response"][:60]}...')
        print()

In [ ]:
if graph:
    lf = CallbackHandler()
    r = graph.invoke({'query': 'No puedo ver mi factura', 'intent': '', 'response': ''},
                     config={'callbacks': [lf]})
    assert r['intent'] == 'finance', f"Esperaba finance, obtuvo: {r['intent']}"
    assert len(r['response']) > 5
    print('Checks E09 OK ✅')